# Introduction

Welcome to Binary Classification with a Software Defects competition! The problem in hands is that we have to predict whether a C program has any defects or not. The metric we will use is Area Under the ROC Curve.

If you want to read the description of the original dataset, you can visit this page: https://www.kaggle.com/datasets/semustafacevik/software-defect-prediction.

# Loading Libraries and Datasets

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from category_encoders import OneHotEncoder, GLMMEncoder, TargetEncoder, CatBoostEncoder
from sklearn import set_config
from sklearn.inspection import permutation_importance
from sklearn.model_selection import StratifiedKFold, RepeatedStratifiedKFold
from sklearn.feature_selection import SequentialFeatureSelector
from sklearn.ensemble import RandomForestRegressor, IsolationForest
from sklearn.metrics import roc_auc_score, roc_curve, make_scorer, f1_score
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import SimpleImputer, IterativeImputer, KNNImputer
from sklearn.metrics.pairwise import euclidean_distances
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.base import BaseEstimator, TransformerMixin, clone
from sklearn.preprocessing import FunctionTransformer, StandardScaler, MinMaxScaler, LabelEncoder
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression, RidgeClassifier
from sklearn.naive_bayes import GaussianNB, BernoulliNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.ensemble import HistGradientBoostingClassifier, GradientBoostingClassifier, AdaBoostClassifier
from sklearn.ensemble import VotingClassifier, StackingClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis, QuadraticDiscriminantAnalysis
from sklearn.gaussian_process import GaussianProcessClassifier
from scipy.cluster.hierarchy import dendrogram, linkage
from scipy.spatial.distance import squareform
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

sns.set_theme(style = 'white', palette = 'viridis')
pal = sns.color_palette('viridis')

pd.set_option('display.max_rows', 100)
set_config(transform_output = 'pandas')
pd.options.mode.chained_assignment = None

In [ ]:
train = pd.read_csv(r'/kaggle/input/playground-series-s3e23/train.csv', index_col = 'id')
test = pd.read_csv(r'/kaggle/input/playground-series-s3e23/test.csv', index_col = 'id')
orig_train = pd.read_csv(r'/kaggle/input/software-defect-prediction/jm1.csv')

# Descriptive Statistics

Let's begin by taking a peek at our training dataset first

In [ ]:
train.head(10)

In [ ]:
desc = pd.DataFrame(index = list(train))
desc['count'] = train.count()
desc['nunique'] = train.nunique()
desc['%unique'] = desc['nunique'] / len(train) * 100
desc['null'] = train.isnull().sum()
desc['type'] = train.dtypes
desc = pd.concat([desc, train.describe().T], axis = 1)
desc

We can see that we have 101k rows and 22 columns, including our target here, which makes it 21 features.

Let's see the test dataset now.

In [ ]:
test.head(10)

In [ ]:
desc = pd.DataFrame(index = list(test))
desc['count'] = test.count()
desc['nunique'] = test.nunique()
desc['%unique'] = desc['nunique'] / len(test) * 100
desc['null'] = test.isnull().sum()
desc['type'] = test.dtypes
desc = pd.concat([desc, test.describe().T], axis = 1)
desc

On the test dataset, we have 67k rows. There is also no missing value on both.

Finally, let's try to see the original dataset.

In [ ]:
orig_train.head(10)

In [ ]:
desc = pd.DataFrame(index = list(orig_train))
desc['count'] = orig_train.count()
desc['nunique'] = orig_train.nunique()
desc['%unique'] = desc['nunique'] / len(orig_train) * 100
desc['null'] = orig_train.isnull().sum()
desc['type'] = orig_train.dtypes
desc = pd.concat([desc, orig_train.describe().T], axis = 1)
desc

Well, we can see that the original dataset have categorical features despite of the fact that they consist of numerical values. Let's take a dive a bit deeper.

In [ ]:
orig_train.uniq_Op.unique()

Look at what we have found! There is a question mark inside the column. Now we know how to preprocess the dataset

# Preprocessing

What we have to do is simply replace `?` character with NaN and then convert the column into float.

In [ ]:
for object_features in list(orig_train.loc[:, orig_train.dtypes == 'O']):
    orig_train[object_features] = orig_train[object_features].replace({'?' : np.nan}).astype('float64')

# Adversarial Validation

The purpose of adversarial validation is to check whether train and test dataset have similar distribution or not. If the validation gives ROC-AUC score of close to .5, we can say that both datasets are similar. However, if it's far from .5, both dataset have different distribution.

The reason we want to do this is to make sure that we can trust our CV score, since a trusted CV only comes from dataset with similar distribution.

In [ ]:
#thanks to @carlmcbrideellis
#https://www.kaggle.com/code/carlmcbrideellis/what-is-adversarial-validation

def adversarial_validation(dataset_1 = train, dataset_2 = test, label = 'Train-Test'):

    adv_train = dataset_1.drop('defects', axis = 1)
    adv_test = dataset_2.copy()

    adv_train['is_test'] = 0
    adv_test['is_test'] = 1

    adv = pd.concat([adv_train, adv_test], ignore_index = True)

    adv_shuffled = adv.sample(frac = 1)

    adv_X = adv_shuffled.drop('is_test', axis = 1)
    adv_y = adv_shuffled.is_test

    skf = StratifiedKFold(n_splits = 5, random_state = 42, shuffle = True)

    val_scores = []
    predictions = np.zeros(len(adv))

    for fold, (train_idx, val_idx) in enumerate(skf.split(adv_X, adv_y)):
    
        adv_lr = XGBClassifier(random_state = 42)
        adv_lr.fit(adv_X.iloc[train_idx], adv_y.iloc[train_idx])
        
        val_preds = adv_lr.predict_proba(adv_X.iloc[val_idx])[:,1]
        predictions[val_idx] = val_preds
        val_score = roc_auc_score(adv_y.iloc[val_idx], val_preds)
        val_scores.append(val_score)
    
    fpr, tpr, _ = roc_curve(adv['is_test'], predictions)
    
    plt.figure(figsize = (10, 10), dpi = 300)
    sns.lineplot(x=[0, 1], y=[0, 1], linestyle="--", label="Indistinguishable Datasets")
    sns.lineplot(x=fpr, y=tpr, label="Adversarial Validation Classifier")
    plt.title(f'{label} Validation = {np.mean(val_scores):.5f}', weight = 'bold', size = 17)
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.show()

In [ ]:
adversarial_validation()

The result is very close to .5, therefore we can trust our CV.

# Distribution of Numerical Features

Now that we have done taking a peek at the descriptive statistics of the datasets and doing adversarial validation, let's try to see the feature distribution this time.

In [ ]:
fig, ax = plt.subplots(7, 3, figsize = (15, 25), dpi = 300)
ax = ax.flatten()

for i, column in enumerate(list(test)):
        
    sns.kdeplot(train[column], ax=ax[i], color=pal[0])
    sns.kdeplot(test[column], ax=ax[i], color=pal[2], warn_singular = False)
    sns.kdeplot(orig_train[column], ax=ax[i], color=pal[1])
    
    ax[i].set_title(f'{column} Distribution', size = 14)
    ax[i].set_xlabel(None)
    
fig.suptitle('Distribution of Feature\nper Dataset\n', fontsize = 24, fontweight = 'bold')
fig.legend(['Train', 'Test', 'Original Train'])
plt.tight_layout()

The train and test datasets have similar distribution as expected, but the original dataset is, well, it's a bit hard to see. One thing for sure is that all features are extremely skewed.

# Target Distribution

We still need to check one last distribution: our target.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize = (16, 5))
ax = ax.flatten()

ax[0].pie(
    train['defects'].value_counts(), 
    shadow = True, 
    explode = [.1 for i in range(train.defects.nunique())], 
    autopct = '%1.f%%',
    textprops = {'size' : 14, 'color' : 'white'}
)

sns.countplot(data = train, y = 'defects', ax = ax[1], palette = 'viridis', order = train['defects'].value_counts().index)
ax[1].yaxis.label.set_size(20)
plt.yticks(fontsize = 12)
ax[1].set_xlabel('Count', fontsize = 20)
ax[1].set_ylabel(None)
plt.xticks(fontsize = 12)

fig.suptitle('Defects Distribution in Train Dataset', fontsize = 25, fontweight = 'bold')
plt.tight_layout()

In [ ]:
fig, ax = plt.subplots(1, 2, figsize = (16, 5))
ax = ax.flatten()

ax[0].pie(
    orig_train['defects'].value_counts(), 
    shadow = True, 
    explode = [.1 for i in range(orig_train.defects.nunique())], 
    autopct = '%1.f%%',
    textprops = {'size' : 14, 'color' : 'white'}
)

sns.countplot(data = orig_train, y = 'defects', ax = ax[1], palette = 'viridis', order = orig_train['defects'].value_counts().index)
ax[1].yaxis.label.set_size(20)
plt.yticks(fontsize = 12)
ax[1].set_xlabel('Count', fontsize = 20)
ax[1].set_ylabel(None)
plt.xticks(fontsize = 12)

fig.suptitle('Defects Distribution in Original Train Dataset', fontsize = 25, fontweight = 'bold')
plt.tight_layout()

We can see above that the target distribution is quite unbalanced: there are less than 25% of defective C codes in both competition dataset and original dataset.

# Correlation

If we want to see the relationship between features, we can try calculating the correlation. If two features have negative correlation, it means that an increase of value in one feature will result in a decrease of value in another feature. On the other hand, positive correlation means that an increase of value in one feature will result in an increase of value in another. Let's try to take a look.

In [ ]:
def heatmap(dataset, label = None):
    corr = dataset.corr(method = 'spearman')
    plt.figure(figsize = (15, 15), dpi = 300)
    mask = np.zeros_like(corr)
    mask[np.triu_indices_from(mask)] = True
    sns.heatmap(corr, mask = mask, cmap = 'viridis', annot = True, annot_kws = {'size' : 7})
    plt.title(f'{label} Dataset Correlation Matrix\n', fontsize = 25, weight = 'bold')
    plt.show()

In [ ]:
heatmap(train, 'Train')
heatmap(test, 'Test')
heatmap(orig_train, 'Original Train')

We can see from above that there are perfectly correlated features. Let's simplify the matrix with hierarchial clustering.

# Hierarchial Clustering

We can see both the strength and the direction of relaionship between features above. However, if that's too many, we can try to cluster the features with hierarchial clustering.

In [ ]:
def distance(data, label = ''):
    #thanks to @sergiosaharovsky for the fix
    corr = data.corr(method = 'spearman')
    dist_linkage = linkage(squareform(1 - abs(corr)), 'complete')
    
    plt.figure(figsize = (10, 8), dpi = 300)
    dendro = dendrogram(dist_linkage, labels=data.columns, leaf_rotation=90)
    plt.title(f'Feature Distance in {label} Dataset', weight = 'bold', size = 20)
    plt.show()

In [ ]:
distance(train, 'Train')
distance(test, 'Test')
distance(orig_train, 'Original Train')

We can try grouping the duplicate features now:
1. `total_Op`, `n`, `v`, and `b`,
2. `t` and `e`, which actually can be included in group 1,
3. `total_Opnd` can also be included in group 1 if you want to stretch it further,
4. `branchCount` and `v(g)`.

Group 1, 2, and 3 belongs to Halstead measures, while group 4 belongs to McCabe metrics.

# Preparation

This is where we start preparing everything if we want to start building machine learning models.

In [ ]:
X = train.copy()
y = X.pop('defects')

seed = 42
splits = 5
skf = StratifiedKFold(n_splits = splits, random_state = seed, shuffle = True)
np.random.seed(seed)

# Model Cross Validation

Let's start by evaluating the performance of our model first. Since we know that the original dataset have missing value, we will use Simple Imputer on our pipeline. We will also concatenate the original dataset only during the cross-validation process for robustness.

In [ ]:
def cross_val_score(estimator, cv = skf, label = '', include_original = False):
    
    X = train.copy()
    y = X.pop('defects')
    
    #initiate prediction arrays and score lists
    val_predictions = np.zeros((len(X)))
    #train_predictions = np.zeros((len(sample)))
    train_scores, val_scores = [], []
    
    #training model, predicting prognosis probability, and evaluating metrics
    for fold, (train_idx, val_idx) in enumerate(cv.split(X, y)):
        
        model = clone(estimator)
        
        #define train set
        X_train = X.iloc[train_idx]
        y_train = y.iloc[train_idx]
        
        #define validation set
        X_val = X.iloc[val_idx]
        y_val = y.iloc[val_idx]
        
        if include_original:
            X_train = pd.concat([X_train, orig_train.drop('defects', axis = 1)])
            y_train = pd.concat([y_train, orig_train.defects])
        
        #train model
        model.fit(X_train, y_train)
        
        #make predictions
        train_preds = model.predict_proba(X_train)[:, 1]
        val_preds = model.predict_proba(X_val)[:, 1]
                  
        val_predictions[val_idx] += val_preds
        
        #evaluate model for a fold
        train_score = roc_auc_score(y_train, train_preds)
        val_score = roc_auc_score(y_val, val_preds)
        
        #append model score for a fold to list
        train_scores.append(train_score)
        val_scores.append(val_score)
    
    print(f'Val Score: {np.mean(val_scores):.5f} ± {np.std(val_scores):.5f} | Train Score: {np.mean(train_scores):.5f} ± {np.std(train_scores):.5f} | {label}')
    
    return val_scores, val_predictions

In [ ]:
score_list, oof_list = pd.DataFrame(), pd.DataFrame()

models = [
    ('log', LogisticRegression(random_state = seed, max_iter = 1000000)),
    ('lda', LinearDiscriminantAnalysis()),
    ('gnb', GaussianNB()),
    ('bnb', BernoulliNB()),
    ('knn', KNeighborsClassifier()),
    ('rf', RandomForestClassifier(random_state = seed)),
    ('et', ExtraTreesClassifier(random_state = seed)),
    ('xgb', XGBClassifier(random_state = seed)),
    ('lgb', LGBMClassifier(random_state = seed)),
    ('dart', LGBMClassifier(random_state = seed, boosting_type = 'dart')),
    ('cb', CatBoostClassifier(random_state = seed, verbose = 0)),
    ('gb', GradientBoostingClassifier(random_state = seed)),
    ('hgb', HistGradientBoostingClassifier(random_state = seed)),
]

for (label, model) in models:
    score_list[label], oof_list[label] = cross_val_score(
        make_pipeline(SimpleImputer(), model),
        label = label,
        include_original = False
    )

In [ ]:
plt.figure(figsize = (8, 4), dpi = 300)
sns.barplot(data = score_list.reindex((-1 * score_list).mean().sort_values().index, axis = 1), palette = 'viridis', orient = 'h')
plt.title('Score Comparison', weight = 'bold', size = 20)
plt.show()

As can be seen above, LightGBM's DART gives the best result. What's surprising is that both Linear Discriminant Analysis and Gaussian Naive Bayes are quite competitive.

# Voting Ensemble

Now let's try to define the weight of each model and then build a voting ensemble. We will use Ridge Classifier to define the weight by fitting it on OOF prediction and the true label.

In [ ]:
weights = RidgeClassifier(random_state = seed).fit(oof_list, train.defects).coef_[0]
pd.DataFrame(weights, index = list(oof_list), columns = ['weight per model'])

After defining the weight, we can start building a Voting Ensemble of our models.

In [ ]:
voter = VotingClassifier(models, weights = weights, voting = 'soft')
_ = cross_val_score(
    make_pipeline(SimpleImputer(), voter),
    include_original = False
)

We can see an improvement in the CV score above!

# Prediction and Submission

Finally, let's train our chosen model on the whole train dataset and do prediction on the test dataset.

In [ ]:
model = make_pipeline(
    SimpleImputer(),
    voter
)

model.fit(X, y)

In [ ]:
submission = test.copy()
submission['defects'] = model.predict_proba(submission)[:, 1]

submission.defects.to_csv('submission.csv')

In [ ]:
plt.figure(figsize = (15, 10), dpi = 300)
sns.kdeplot(submission.defects, fill = True)
plt.title("Distribution of Defects Probability", weight = 'bold', size = 25)
plt.show()

Thanks for reading!